In [1]:
#%pip uninstall -y av faster-whisper yt-dlp
%pip install faster-whisper
%pip install -U --pre "yt-dlp[default]" --break-system-packages
%pip install tqdm

%pip install "av<17"

import av
print(f"av version after re-installing faster-whisper: {av.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 67.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 100.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 39.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [faster-whisper]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 93.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mutagen]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 59.1 MB/s  0:00:00
  Attempting uninstall: av
    Found existing installation: av 19.0.2
    Uninstalling av-19.0.2:
      Successfully uninstalled av-19.0.2
av version after re-installing faster-whisper: 16.1.0


In [2]:
!nvidia-smi

Tue Nov 17 21:11:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
VIDEO_URL = 'https://www.bilibili.com/video/BV1RosT6CEad/'

In [4]:
# yt-dlp specs
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "yt_dlp", "--version"],
    capture_output=True, text=True, check=True,
)
print(result.stdout.strip())


2026.11.16.232655


In [5]:
%pip install nvidia-cublas-cu12 nvidia-cudnn-cu12

import os
# Add the paths to the installed nvidia packages to LD_LIBRARY_PATH
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib"

In [6]:
!ln -sf /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12 /usr/lib/libcublas.so.12
!ln -sf /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublasLt.so.12 /usr/lib/libcublasLt.so.12
!ln -sf /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn.so.9 /usr/lib/libcudnn.so.9

In [ ]:
print("starting...")
import os
from faster_whisper import WhisperModel
print("imported...")

# Ensure environment variables are loaded in this execution context
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib"

# Define the path in your Google Drive where models will be stored
model_cache_dir = '/content/drive/MyDrive/faster_whisper_models'

# Create the directory if it doesn't exist
os.makedirs(model_cache_dir, exist_ok=True)

model_size = "large-v3"
model = WhisperModel(
    model_size,
    device="cuda",
    compute_type="float16",
    download_root=model_cache_dir
)
print(f"{model_size} model loaded ...")

starting...
imported...


In [ ]:
import os
import subprocess
import yt_dlp


# Replace this with the video or audio URL to transcribe.
url = VIDEO_URL
download_dir = "/content/downloads"
os.makedirs(download_dir, exist_ok=True)

# We revert back to a clean setup with standard headers to avoid API type assertion issues.
# Note: If Bilibili still returns a 412 error on this cloud IP, you will need to upload a 'cookies.txt'
# file from your browser and uncomment the cookiefile line below.

ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": os.path.join(download_dir, "%(title)s.%(ext)s"),
    "noplaylist": True,
    # "cookiefile": "/content/cookies.txt",  # Uncomment this and upload cookies.txt if 412 error persists
    "http_headers": {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Referer': 'https://www.bilibili.com/'
        #"Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    }
}
'''
ydl_opts = {
    "format": "bestaudio/best",
    "outtmpl": os.path.join(download_dir, "%(title)s.%(ext)s"),
    "noplaylist": True,
    "impersonate": yt_dlp.networking.impersonate.ImpersonateTarget("chrome"),
    "http_headers": {"Referer": "https://www.bilibili.com/"},
}
'''
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=True)
    downloaded_filename = ydl.prepare_filename(info)

filename = os.path.splitext(downloaded_filename)[0] + ".wav"
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        downloaded_filename,
        "-vn",
        "-acodec",
        "pcm_s16le",
        filename,
    ],
    check=True,
)

print(f"Downloaded source: {downloaded_filename}")
print(f"Converted audio: {filename}")

In [ ]:
import os
from tqdm.auto import tqdm
import textwrap
import av
from faster_whisper import WhisperModel

# Force set the LD_LIBRARY_PATH environment variable
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib"

# Re-initialize the model in this context to ensure it binds to the new library paths
model_cache_dir = '/content/drive/MyDrive/faster_whisper_models'
os.makedirs(model_cache_dir, exist_ok=True)

model_size = "large-v3"
model = WhisperModel(
    model_size,
    device="cuda",
    compute_type="float16",
    download_root=model_cache_dir
)

translated_subtitlename = os.path.splitext(filename)[0] + ".en.srt"

print(f"av version being used: {av.__version__}")
print("Translating Japanese audio to English...")

# Start transcription
english_segment_generator, translation_info = model.transcribe(
    filename,
    language="ja",
    task="translate",
    beam_size=5,
    temperature=0.0,
    condition_on_previous_text=False,
    no_speech_threshold=0.6,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=500),
    word_timestamps=True
)

total_duration = getattr(translation_info, "duration", None)
progress = tqdm(
    total=total_duration,
    unit="sec",
    desc="Translating subtitles",
    dynamic_ncols=True,
)

english_segments = []
last_end = 0.0
for segment in english_segment_generator:
    english_segments.append(segment)
    progress.update(max(0.0, segment.end - last_end))
    last_end = segment.end
progress.close()

In [ ]:
# Edit subs
# Functions definitions
def chunk_segments_by_words(segments, max_chars, max_lines):
    """Re-splits segments into shorter subtitle chunks using word timestamps."""
    new_segments = []
    for segment in segments:
        words = segment.words  # requires word_timestamps=True
        if not words:
            new_segments.append(segment)
            continue

        current_words = []
        current_len = 0
        chunk_start = words[0].start

        def flush(end_time):
            text = "".join(w.word for w in current_words).strip()
            new_segments.append(type(segment)(
                start=chunk_start, end=end_time, text=text,
                words=current_words[:], id=0, seek=0, tokens=[],
                avg_logprob=0, no_speech_prob=0, compression_ratio=0, temperature=0
            ))

        for w in words:
            wlen = len(w.word)
            if current_len + wlen > max_chars and current_words:
                flush(current_words[-1].end)
                current_words, current_len = [], 0
                chunk_start = w.start
            current_words.append(w)
            current_len += wlen

        if current_words:
            flush(current_words[-1].end)

    return new_segments

def format_timestamp(seconds):
    """Converts seconds to SRT format (00:00:08,850)."""
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hrs:02d}:{mins:02d}:{secs:02d},{millis:03d}"



def write_srt(segments, output_filename, max_width):
    """Writes segments to SRT with word wrapping for readability."""
    with open(output_filename, "w", encoding="utf-8") as f:
        for i, segment in enumerate(segments, start=1):
            start_time = format_timestamp(segment.start)
            end_time = format_timestamp(segment.end)

            # Wrap text to standard subtitle width
            wrapped_text = textwrap.fill(segment.text.strip(), width=max_width)

            f.write(f"{i}\n")
            f.write(f"{start_time} --> {end_time}\n")
            f.write(f"{wrapped_text}\n\n")


def enforce_min_duration(segments, min_duration=1.0, min_gap=0.05):
    """Extends short subtitle segments to a minimum duration without overlapping the next one."""
    segments = list(segments)
    for i, seg in enumerate(segments):
        duration = seg.end - seg.start
        if duration < min_duration:
            new_end = seg.start + min_duration
            # don't overlap the next segment's start
            if i + 1 < len(segments):
                next_start = segments[i + 1].start
                new_end = min(new_end, next_start - min_gap)
            new_end = max(new_end, seg.start + 0.1)  # never shrink below original
            segments[i] = type(seg)(
                start=seg.start, end=new_end, text=seg.text,
                words=seg.words, id=0, seek=0, tokens=[],
                avg_logprob=0, no_speech_prob=0, compression_ratio=0, temperature=0
            )
    return segments


# Edit segments
english_segments = chunk_segments_by_words(english_segments, max_chars=42, max_lines=2)
print('chunk_segments_by_words done')

english_segments = enforce_min_duration(english_segments, min_duration=0.5)
print('enforce_min_duration done')

# Write to file
write_srt(english_segments, translated_subtitlename, max_width=42)
print('write_srt done')
print(f"English subtitles saved with wrapping: {translated_subtitlename}")
